# Quick look at the data

Before writing the project brief I wanted real numbers in front of me instead of guesses, so this notebook just loads the CSVs and pulls a few totals. No cleaning yet, that comes later.

Data: *The Complete Journey* by dunnhumby, downloaded from Kaggle into `data/raw/`. I'm leaving out `causal_data.csv` (the promotions file) for now because it's ~700 MB and I don't need it yet.

In [1]:
import pandas as pd

raw = "../data/raw/"

tx = pd.read_csv(raw + "transaction_data.csv")
campaigns = pd.read_csv(raw + "campaign_table.csv")
campaign_desc = pd.read_csv(raw + "campaign_desc.csv")
demo = pd.read_csv(raw + "hh_demographic.csv")

# the original files mix UPPER and lower case column names, so I'm making them all lowercase
for df in [tx, campaigns, campaign_desc, demo]:
    df.columns = df.columns.str.lower()

print(tx.shape)
tx.head()

(2595732, 12)


,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0


There's no real date column. `day` is a day number (day 1 = first day of the data) and `week_no` is the week number, so I'll use those for anything time-based.

## Sales

I'm splitting total sales into **households × trips per household × spend per trip**. If those three multiply back to the total, the numbers are consistent. It also means that later on I can say whether a change in sales came from more customers, more visits or bigger baskets.

In [2]:
total_sales = tx["sales_value"].sum()
households = tx["household_key"].nunique()
trips = tx["basket_id"].nunique()      # one basket = one shopping trip

trips_per_household = trips / households
spend_per_trip = total_sales / trips

print("Days covered:", tx["day"].min(), "to", tx["day"].max())
print("Weeks covered:", tx["week_no"].min(), "to", tx["week_no"].max())
print("Stores:", tx["store_id"].nunique())
print(f"Total sales: ${total_sales:,.0f}")
print(f"Households: {households:,}")
print(f"Trips: {trips:,}")
print(f"Trips per household: {trips_per_household:.1f}")
print(f"Spend per trip: ${spend_per_trip:.2f}")

Days covered: 1 to 711
Weeks covered: 1 to 102
Stores: 582
Total sales: $8,057,463
Households: 2,500
Trips: 276,484
Trips per household: 110.6
Spend per trip: $29.14


In [3]:
# sanity check: does households x trips x spend give back total sales?
rebuilt = households * trips_per_household * spend_per_trip
print(f"${rebuilt:,.0f}", "matches" if round(rebuilt) == round(total_sales) else "DOESN'T MATCH")

$8,057,463 matches


## Discounts

In this version of the data, discounts are stored as negative numbers. Checking that before adding anything up.

In [4]:
tx[["retail_disc", "coupon_disc"]].describe()

,retail_disc,coupon_disc
count,2.595732e+06,2.595732e+06
mean,-5.387054e-01,-1.641600e-02
std,1.249191e+00,2.168410e-01
min,-1.800000e+02,-5.593000e+01
25%,-6.900000e-01,0.000000e+00
50%,-1.000000e-02,0.000000e+00
75%,0.000000e+00,0.000000e+00
max,3.990000e+00,0.000000e+00


In [5]:
# abs() turns the negatives into positive dollar amounts
retail_disc = tx["retail_disc"].abs().sum()
coupon_disc = tx["coupon_disc"].abs().sum()
any_discount = (tx["retail_disc"] != 0) | (tx["coupon_disc"] != 0)

print(f"Retail discounts: ${retail_disc:,.0f}")
print(f"Coupon discounts: ${coupon_disc:,.0f}")
print(f"Discounts as % of sales: {(retail_disc + coupon_disc) / total_sales:.1%}")
print(f"Purchase lines with some discount: {any_discount.mean():.0%}")

Retail discounts: $1,398,352
Coupon discounts: $42,612
Discounts as % of sales: 17.9%
Purchase lines with some discount: 51%


About 18% of sales went back out as discounts ($1.44M on $8.06M of sales), and half of everything people bought had some discount on it. Almost all of it is the store's own price cuts. Coupons are only about $43K.

That's the reason for this project. If half of all purchases are discounted, I want to know how much of that is actually bringing in extra sales, and how much is just a lower price on things people would have bought anyway.

## Campaigns and demographics

In [6]:
reached = campaigns["household_key"].nunique()

print("Campaigns:", campaign_desc["campaign"].nunique())
print(campaign_desc["description"].value_counts().sort_index())
print(f"Households that got at least one campaign: {reached:,} of {households:,} ({reached / households:.0%})")
print(f"Households with demographic info: {demo['household_key'].nunique():,} of {households:,}")

Campaigns: 30
description
TypeA     5
TypeB    19
TypeC     6
Name: count, dtype: int64
Households that got at least one campaign: 1,584 of 2,500 (63%)
Households with demographic info: 801 of 2,500


## Things to remember for the brief

- Two years of data: days 1 to 711, weeks 1 to 102. There are no real dates, only day and week numbers.
- 2,500 households across 582 stores, about 276K shopping trips. That works out to roughly one trip a week per household.
- Demographics only exist for 801 households (about a third), so anything I say about age or income groups only covers them.
- 916 households never got a campaign. They could be the comparison group for the campaign analysis, but they probably weren't picked at random, so I need to check that first.
- `retail_disc` should always be negative, but the max is +3.99, so a few rows have positive values. Need to look at these in the cleaning step.